# Forecasting the future: what a regime model knows, and for how long

A hidden Markov model over growth, inflation and interest rates finds a handful of
persistent economic regimes. Conditioning on which regime the economy is in, and
projecting that regime forward, gives forecasts for ten binary indicators at one,
five and ten years.

This report says what those regimes are, how the forecasts were scored against a
rule written down before they were made, and where the method runs out of
information. That last part is the finding the project was built to produce, and
it is not a disappointment: a method that knows the horizon beyond which it has
nothing to say is more useful than one that does not.

In [ ]:
import pandas as pd

from economic_regime_forecasting.configuration.registry import load_registries
from economic_regime_forecasting.configuration.run_settings import (
    ARTIFACTS,
    DEFAULT_RUN_SETTINGS,
)
from economic_regime_forecasting.data.cache import ArtifactStore, SeriesCache
from economic_regime_forecasting.data.panel import assemble_point_in_time_panel, load_final_series
from economic_regime_forecasting.features.observation_matrix import build_observation_matrix
from economic_regime_forecasting.models.gaussian_hidden_markov_model import (
    GaussianHiddenMarkovModel,
)
from economic_regime_forecasting.models.regime_forecast import transition_matrix_table
from economic_regime_forecasting.reporting import figures

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

settings = DEFAULT_RUN_SETTINGS
registry, indicators = load_registries()
cache = SeriesCache(settings.cache.raw, settings.cache.vintage)
artifacts = ArtifactStore(settings.cache.models)

model = GaussianHiddenMarkovModel.from_dictionary(artifacts.read_json(ARTIFACTS.selected_model))
regimes = artifacts.read_table(ARTIFACTS.regime_descriptions)
mixing = artifacts.read_table(ARTIFACTS.mixing_diagnostics)
metrics = artifacts.read_table(ARTIFACTS.evaluation_metrics)
verdicts = artifacts.read_table(ARTIFACTS.verdicts)
forecasts = artifacts.read_table(ARTIFACTS.current_forecasts)

## 1. The regimes

Five of them, found rather than imposed. The model was fitted on continuous
standardised growth, inflation and interest rates; the labels below are read off
the fitted emission means afterwards, so "stagflation" is an output of the fit and
not an assumption fed into it.

In [ ]:
regimes[
    [
        "state",
        "regime",
        "growth_natural",
        "inflation_natural",
        "rates_natural",
        "population_share",
        "expected_duration_months",
    ]
].round(3)

In [ ]:
transition_matrix_table(model, [f"{index}" for index in regimes["state"]]).round(4)

Every diagonal entry is above 0.94, which is what makes these regimes rather than
weather. The shortest expected visit is well over a year.

In [ ]:
today = pd.Timestamp.today().date()
matrix = build_observation_matrix(assemble_point_in_time_panel(registry, today, cache), registry)
filtered = model.filtered_state_probabilities(matrix.values)
recession = load_final_series(registry, cache, ["recession_indicator"])["recession_indicator"]
figures.plot_regime_timeline(
    matrix.dates, filtered, [str(name) for name in regimes["short_regime"]], recession
)

## 2. The information horizon

This is the central result and it is a property of the method rather than of the
data. A regime transition matrix mixes: the distance between a projected regime
distribution and the model's long-run distribution decays geometrically, at a rate
set by the transition matrix's second largest eigenvalue.

Past some horizon the projection *is* the long-run distribution, which means the
forecast *is* the unconditional base rate. The model can still be right; it just is
not saying anything a base rate does not already say. Presenting that as a
prediction would misdescribe it even when it scores well, which is why one of the
five acceptance gates tests for it directly.

In [ ]:
print(f"second largest eigenvalue modulus {model.second_largest_eigenvalue_modulus():.4f}")
mixing.round(4)

In [ ]:
figures.plot_mixing(mixing, settings.information_horizon_total_variation_threshold)

## 3. Did it beat the base rate?

The benchmark is an expanding climatology: at each date, the average of the
outcomes that had already resolved by then. It never knows anything the model
could not have known.

Intervals come from a moving-block bootstrap with blocks as long as the horizon.
This matters more than it sounds. Monthly forecasts at a ten-year horizon overlap
by a hundred and nineteen months, so an ordinary bootstrap treats one long episode
as hundreds of independent observations and produces intervals narrow enough to
declare skill that is not there. The effective independent sample size is printed
next to every interval; at ten years it is a single-digit number.

In [ ]:
verdicts.round(4)

In [ ]:
figures.plot_skill_by_horizon(verdicts)

In [ ]:
for horizon in settings.forecast_horizons_in_months:
    display(figures.plot_skill_by_indicator(metrics, horizon))

## 4. What ships

A horizon ships the model only if all five pre-registered gates hold. Otherwise it
ships the climatological base rate and names the gate that failed. A well
calibrated base rate beats an overconfident model under Brier scoring, so this is
a result rather than a retreat.

In [ ]:
submission = pd.read_csv("../submission/forecasts.csv")
submission.pivot(index="indicator", columns="horizon_years", values="probability")

In [ ]:
submission[
    [
        "indicator",
        "horizon_years",
        "probability",
        "source",
        "model_probability",
        "climatological_base_rate",
    ]
].round(3)

## 5. What this method cannot do

Stated plainly, because a forecast without its limits is worth less than no
forecast.

**It cannot see past its mixing time.** The information horizon above is not a
tuning problem. Fitting more regimes, or a longer sample, moves it by months, not
by decades. Any method whose long-horizon forecast is a projected Markov chain has
this property.

**Long horizons have almost no independent evidence.** Fifty-five years of monthly
ten-year forecasts contain roughly five independent observations. Every ten-year
statement here rests on about five non-overlapping windows, and no amount of
statistical machinery creates more.

**Questions phrased relative to today are out of scope.** "Will unemployment rise
two points from where it is now" depends on today's level as well as on the
regime, which needs a conditional distribution of the level given the regime
rather than an event probability. The ten indicators are all conditions on a
single month for that reason, and the exclusion is recorded in the registry.

**The consumer price index has no usable archival vintage before 1997** through
the keyless endpoint, so forecasts before then were made on the publication-lag
fallback: correct timing, revised values. The size of that approximation is
measured in notebook one.

**The any-time composition carries an assumption.** It treats whether a condition
holds as depending on last month's condition and this month's regime, and nothing
else. If that is wrong the error shows up as miscalibration, which the calibration
gate tests for; it is not hidden.

**Conditions are read from final data with timing enforced.** For market rates and
recession dating this is exact. For industrial production it is an approximation,
and notebook one measures it.

## 6. Two corrections made after seeing results, both disclosed

Nothing here was tuned toward a positive result, and two changes were made after
the first run that a sceptical reader is entitled to weigh.

**The calibration standard error** treated overlapping monthly forecasts as
independent, which made it about three and a half times too small at one year.
Correcting it, using the block length the pre-registration already requires for
the skill interval, changed the one-year verdict from ship-base-rate to
ship-model. No threshold moved. It was later extended to the correlation between
the ten indicators, measured rather than assumed. ADR 0006 records it including
the argument against it, and every run prints what the uncorrected computation
would have said.

**Five defects in the statistical core** were found by an adversarial review after
the first complete run, recorded in ADR 0007. The one that moved a number made the
long-horizon result *more* adverse to the model: the bootstrap statistic had been
scoring whichever indicators happened to be scoreable on each resample, and fixing
it moved the ten-year skill interval from one that spanned zero to one lying
entirely below it.

## 7. Reproducing this

```bash
poetry install
poetry run forecast fetch-data
poetry run forecast check-gates
poetry run forecast submit
```

Two runs of the same configuration produce identical output. The seed and the
configuration hash are on every result row and in `submission/manifest.json`.